**Running this on Google Colab.** The first code cell fetches this entry's files and installs the packages it was run with, pinned for Colab's Python version. The results in the registry come from the entry's own environment ([requirements.lock.txt](requirements.lock.txt), Python 3.13); a different Python or platform can change the last digits of some numbers.

In [ ]:
# On Google Colab: fetch this entry's files and install its pinned packages. Elsewhere this cell does nothing.
import sys
if "google.colab" in sys.modules:
    import os, subprocess
    if not os.path.isdir("/content/encore"):
        subprocess.run(["git", "clone", "-q", "--depth", "1", "--branch", "colab", "--filter=blob:none", "--sparse",
                        "https://github.com/encorescience/encore-registry", "/content/encore"], check=True)
        subprocess.run(["git", "-C", "/content/encore", "sparse-checkout", "set", "entries/pakhomov-2023"], check=True)
    os.chdir("/content/encore/entries/pakhomov-2023")
    sys.path.insert(0, os.getcwd())
    version = f"{sys.version_info.major}.{sys.version_info.minor}"
    lock = f"requirements.colab-{version}.txt"
    if not os.path.exists(lock):
        print(f"No lock for Python {version}: installing the entry's unpinned requirements (requirements.in).")
        lock = "requirements.in"
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "uv"], check=True)
    subprocess.run(["uv", "pip", "install", "--system", "-q", "-r", lock], check=True)
    print("Installed. If Colab asks to restart the session, restart it and run the notebook from the top again.")


# Reproducing Pakhomov et al. (2023) from dandisets 000686, 000875, 001025, 001027 and 001028

Paper: Pakhomov, Gudvangen, Mangalanathan, Kondratiev, Redondo & Semenov (2023), "Next
generation CANCAN focusing for remote stimulation by nanosecond electric pulses",
Bioelectrochemistry 152, 108437, doi:10.1016/j.bioelechem.2023.108437.

Claims: `claims/pakhomov-2023.yaml`, written from the accepted author manuscript in PubMed
Central (PMC10247516) and checked by a second reader. The paper has no supplement and links no
code.

Datasets, each pinned at its only published version (each dandiset has one published version
besides the draft, with the same assets as the draft; the paper cites no version):

| Dandiset | Version | Paper figure (dataset title) |
|---|---|---|
| 000686 | 0.240515.1909 | Fig. 5, 40 packets against 40 trains, rotated and not |
| 000875 | 0.240515.1925 | Fig. 6, 1Px4, 5Px4 and 10Px4 |
| 001025 | 0.240605.1826 | Fig. 7, voltages in the small (1Px4) and large (10Px4) arrays |
| 001027 | 0.240605.1859 | Fig. 8, Packets 1, 2 and 3 |
| 001028 | 0.240605.1954 | Fig. 9, electrodes 0 to 3 mm above the cells |

Each NWB file holds one dish: one stitched image of YO-PRO-1 (YP) fluorescence. The files hold
no ROI, no pixel size and no electrode positions, so the measurement of Methods 2.4 is rebuilt
from the images (helper `cancan.py`, which says where each parameter comes from).

| Claim | Tested or blocked |
|---|---|
| fig3-field-simulation | blocked, data_missing (no simulated field maps in any dataset) |
| fig4-sham-no-uptake | blocked, data_missing (no single-cell time lapse in any dataset) |
| fig4-unipolar-strongest | blocked, data_missing |
| fig4-bipolar-weaker | blocked, data_missing |
| fig4-twenty-fold-cancellation | blocked, data_missing |
| fig5a-trains-stronger | tested, 000686 |
| fig5a-packet-center-enhanced | tested, 000686 |
| fig5b-rotation-center-peak | tested, 000686; blocked after testing, methods_underspecified |
| fig5c-normalized-contrast | tested, 000686; blocked, depends on fig5b-rotation-center-peak |
| fig6a-more-packets-more-uptake | tested, 000875 |
| fig6b-peak-relative-to-periphery | tested, 000875 |
| fig6-trough-half-of-center | tested, 000875 |
| fig6-peak-narrower | tested, 000875 ; blocked after testing, methods_underspecified |
| fig7a-amplitude-scales-uptake | tested, 001025 |
| fig7b-targeting-independent-of-amplitude | tested, 001025; blocked after testing, methods_underspecified |
| fig7cd-holds-for-large-array | tested, 001025 ; blocked after testing, methods_underspecified |
| discussion-independent-of-array-size | blocked, depends on fig7b-targeting-independent-of-amplitude |
| fig8a-equal-center-uptake | tested, 001027 |
| fig8a-more-uptake-near-electrodes | tested, 001027 |
| fig8b-packet2-best-targeting | tested, 001027 |
| fig8-cancellation-predicts-targeting | blocked, data_missing (rests on the Fig. 4 data) |
| fig9a-gap-reduces-periphery-more | tested, 001028 |
| fig9b-gap-boosts-targeting | tested, 001028 |
| fig9c-restricted-to-center | tested, 001028 |

In [ ]:
import os
os.environ.setdefault("OMP_NUM_THREADS", "1")
import time
import resource
import itertools
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
import cancan as cc

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 30)
pd.set_option("display.max_rows", 200)
T0 = time.time()

## Loading

Every file of the five dandisets is streamed (about 70 files, 3.6 to 11.7 GB per dandiset),
two at a time, and reduced at once to block means of 8 x 8 pixels (5.2 um at the assumed
0.65 um per pixel). The electrode square of each dish is then found in the reduced image.

In [ ]:
DATA = {}
for ds in cc.VERSIONS:
    t = time.time()
    DATA[ds] = cc.load_dandiset(ds)
    print(ds, len(DATA[ds]), "files", f"{time.time() - t:.0f} s",
          f"peak memory {resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / 1e6:.1f} GB")
ALL = [r for ds in DATA for r in DATA[ds]]
for r in ALL:
    r["cond"] = cc.condition(r)
    r["geom_fp"] = cc.find_footprints(r["frames"][0], r["cond"]["array"])
    r["geom_marks"] = None
    if len(r["frames"]) > 1:
        side = 1330.0 if r["cond"]["array"] == "large" else 715.0
        r["geom_marks"] = cc.find_marks(r["frames"][1], side)
    # Electrodes 2 or 3 mm above the cells leave no footprint in the YP image (Results 3.5);
    # their square comes from the needle marks of the second channel.
    r["geom"] = r["geom_marks"] if r["cond"]["gap_mm"] >= 2 else r["geom_fp"]
    r["prof"] = cc.dish_profiles(r["frames"][0], r["geom"], r["cond"]["array"])
print(f"loaded and measured in {time.time() - T0:.0f} s")

## Data check

In [ ]:
rows = []
for r in ALL:
    c = r["cond"]
    rows.append(dict(dandiset=r["dandiset"], file=r["path"].split("/")[-1][:48],
                     subject=r["subject_id"], protocol=c["protocol"], packet=c["packet"],
                     kV=c["voltage_kv"], gap_mm=c["gap_mm"], array=c["array"],
                     frames=r["shape"][0], size=f"{r['shape'][1]}x{r['shape'][2]}",
                     dtype=r["dtype"], min=min(s["min"] for s in r["stats"]),
                     max=max(s["max"] for s in r["stats"]),
                     saturated=max(s["saturated"] for s in r["stats"]),
                     resolution=r["resolution"], unit=r["unit"], start=r["start"][:16]))
inv = pd.DataFrame(rows)
print(inv.drop(columns=["file"]).to_string())
print()
print("Dishes per condition:")
print(inv.groupby(["dandiset", "array", "protocol", "packet", "kV", "gap_mm"]).size().to_string())
print()
print("species:", sorted({r["species"] for r in ALL}), " strain:", sorted({r["strain"] for r in ALL}))
print("series:", sorted({tuple(r["series"]) for r in ALL}))
print("image descriptions:")
for t in sorted({str(r["image_description"]) for r in ALL}):
    print("  -", t[:300])

In [ ]:
# Recordings deposited more than once: identical full-resolution pixel data (SHA-1 of the
# uint16 frame).
by_hash = {}
for r in ALL:
    for i, h in enumerate(r["pixel_sha1"]):
        by_hash.setdefault(h, []).append((r["dandiset"], r["path"].split("/")[-1], i))
dups = {h: v for h, v in by_hash.items() if len(v) > 1}
print(len(dups), "frames deposited more than once:")
for h, v in dups.items():
    print("  ", h[:12], v)
same_ds = [v for v in dups.values() if len({x[0] for x in v}) < len(v)]
print("duplicates inside one dandiset:", len(same_ds))

In [ ]:
# Electrode squares, and the pixel size checked against the paper's geometry: needles 1.5 mm
# across (Methods 2.3), large-array diagonal 8.4 mm (Results 3.5).
g = []
for r in ALL:
    m = cc.geometry_mm(r["geom"])
    g.append(dict(dandiset=r["dandiset"], subject=r["subject_id"][-14:], array=r["cond"]["array"],
                  gap=r["cond"]["gap_mm"], method=r["geom"]["method"],
                  fit_error=round(r["geom"]["err"], 3), contrast=round(r["geom"]["score"], 2),
                  side_mm=round(m["side_mm"], 2), diag_centres_mm=round(m["diag_centres_mm"], 2),
                  diag_inner_mm=round(m["diag_inner_edges_mm"], 2),
                  footprint_mm=round(m["footprint_diameter_mm"], 2),
                  background=round(r["prof"][("a", "w1", "b")]["background"], 0)))
geo = pd.DataFrame(g)
print(geo.to_string())
print()
print(geo.groupby("array")[["side_mm", "diag_centres_mm", "diag_inner_mm", "footprint_mm"]]
      .describe().T.round(2).to_string())

In [ ]:
# The two ways of finding the square, compared where both are possible (electrodes touching the
# cells or 1 mm above them, with a second channel).
cmp = []
for r in ALL:
    if r["geom_marks"] is None:
        continue
    cf_ = r["geom_fp"]["P"].mean(0)
    cm_ = r["geom_marks"]["P"].mean(0)
    cmp.append(dict(dandiset=r["dandiset"], subject=r["subject_id"], gap=r["cond"]["gap_mm"],
                    footprint_fit_error=round(r["geom_fp"]["err"], 3),
                    footprint_contrast=round(r["geom_fp"]["score"], 2),
                    mark_distance_mm=round(r["geom_marks"]["mark_distance"] * cc.UM_PER_PX / 1000, 2),
                    centre_difference_mm=round(np.linalg.norm(cf_ - cm_) * cc.UM_PER_PX / 1000, 2),
                    used=r["geom"]["method"]))
cmp = pd.DataFrame(cmp)
print(cmp.to_string())
ok = cmp[(cmp.gap < 2) & (cmp.footprint_contrast >= 0.4)]
print(f"\nwhere the footprints are clear (contrast >= 0.4, gap < 2 mm, n = {len(ok)}): centre "
      f"difference median {ok.centre_difference_mm.median():.2f} mm, max {ok.centre_difference_mm.max():.2f} mm")

In [ ]:
# Electrodes 2 and 3 mm above the cells: the square from the needle marks, checked by eye. The
# second channel of each 001028 dish (reduced 4 times more), with the square from the needle
# marks (cyan, a red circle of the footprint radius at each corner; dish 08 has no usable marks)
# and the square from the footprint search on the YP image (yellow, dashed).
G28 = sorted(DATA["001028"], key=lambda r: (r["cond"]["gap_mm"], r["subject_id"]))
fig, axs = plt.subplots(2, 6, figsize=(21, 7.6))
for ax, r in zip(axs.ravel(), G28):
    if len(r["frames"]) < 2:
        ax.axis("off"); continue
    s = r["frames"][1][::4, ::4]
    v = s[s > 0]
    ax.imshow(s, cmap="gray", vmin=np.percentile(v, 1), vmax=np.percentile(v, 99.5))
    gm, gf = r["geom_marks"], r["geom_fp"]
    if gm is not None:
        Pm = gm["P"] / 4
        ax.plot(*np.r_[Pm, Pm[:1]].T, "c-", lw=0.8)
        for p in Pm:
            ax.add_patch(plt.Circle(p, gm["r"] / 4, fill=False, color="r", lw=0.8))
    Pf = gf["P"] / 4
    ax.plot(*np.r_[Pf, Pf[:1]].T, "y--", lw=0.8)
    ax.set_title(f"{r['subject_id']} gap {r['cond']['gap_mm']:.0f} mm, used: {r['geom']['method'][:16]}", fontsize=8)
    ax.axis("off")
plt.tight_layout(); plt.show()
# The centre region (inner 1 mm) under the two geometries, for the lifted dishes: the profile from
# the needle-mark square against the profile from the footprint search.
rows = []
for r in G28:
    if r["cond"]["gap_mm"] < 1 or r["geom_marks"] is None:
        continue
    pm_ = cc.dish_profiles(r["frames"][0], r["geom_marks"], "large")[("a", "w1", "a")]
    pf_ = cc.dish_profiles(r["frames"][0], r["geom_fp"], "large")[("a", "w1", "a")]
    hs = cc.half_set("a")
    vm, vf = cc.dish_values(pm_, hs, "a", "a"), cc.dish_values(pf_, hs, "a", "a")
    inner = vm["d"] <= 1.0
    rows.append(dict(subject=r["subject_id"], gap=r["cond"]["gap_mm"],
                     centre_shift_mm=np.linalg.norm(r["geom_marks"]["P"].mean(0) - r["geom_fp"]["P"].mean(0)) * cc.UM_PER_PX / 1000,
                     centre_marks=vm["centre"], centre_footprints=vf["centre"],
                     inner_mean_marks=np.nanmean(vm["profile"][inner]), inner_mean_footprints=np.nanmean(vf["profile"][inner]),
                     norm_centre_marks=vm["norm"][0], norm_centre_footprints=vf["norm"][0],
                     peak_mm_marks=vm["d"][int(np.nanargmax(vm["profile"]))], peak_mm_footprints=vf["d"][int(np.nanargmax(vf["profile"]))]))
print(pd.DataFrame(rows).round(2).to_string())

In [ ]:
# Every dish with its electrode square and the ROI diagonals (reduced 4 times more for display).
n = len(ALL)
fig, axs = plt.subplots(int(np.ceil(n / 8)), 8, figsize=(24, 3.3 * int(np.ceil(n / 8))))
for ax, r in zip(axs.ravel(), ALL):
    s = r["frames"][0][::4, ::4]
    v = s[s > 0]
    ax.imshow(s, cmap="gray", vmin=np.percentile(v, 1), vmax=np.percentile(v, 99.5))
    P = r["geom"]["P"] / 4
    for p in P:
        ax.add_patch(plt.Circle(p, r["geom"]["r"] / 4, fill=False, color="r", lw=0.8))
    c = P.mean(0)
    R = cc.HALF_RANGE_MM[r["cond"]["array"]] * 1000 / cc.UM_PER_PX / 4
    for p in P:
        u = (p - c) / np.linalg.norm(p - c)
        ax.plot([c[0], c[0] + u[0] * R], [c[1], c[1] + u[1] * R], "c-", lw=0.8)
    ax.set_title(f"{r['dandiset']} {r['subject_id'][-12:]}\n{r['cond']['protocol']} "
                 f"P{r['cond']['packet']} {r['cond']['voltage_kv']} kV {r['cond']['gap_mm']:.0f} mm",
                 fontsize=7)
    ax.axis("off")
for ax in axs.ravel()[n:]:
    ax.axis("off")
plt.tight_layout()
plt.show()

There is no clock to check: each file is one image taken after the treatment, with no trial
table or time series. The treatment of each dish is read from its subject id (000686, 000875)
or its session description (001025, 001027, 001028); array size is not stated in any file
and is read from the electrode spacing above.

The needle-mark square of the lifted electrodes (second cell above the dish overview) cannot be
checked against the footprints at 2 and 3 mm, where the footprint search finds no clear
footprint (contrast 0.09 to 0.17 in the comparison table). Where both are run on a lifted dish,
the mean of the inner 1 mm differs by at most 29% between the two squares, but the single
centre ROI can differ by a factor of up to 2.7 (dish 05, 0.17 mm apart): values read from the
centre ROI alone depend on where the centre is placed.

## Tools for the claims

`group_values` (helper) gives a group's values under one combination of the claims file's
readings O1 to O7 and O9. Numbers are reported under the main reading (`cc.PRIMARY`, the first
reading of each option: 92 or 224 ROIs per diagonal, square ROIs, both diagonals, the ROI on the
centre, the mean of the end ROIs, per-dish normalization, raw intensity and dish means).

A test is run under every combination of the readings its criterion names, and its outcome
follows the claims file's rule: the outcome all readings give. Where they differ, the outcome
is the one all the `supported` readings give (the reason for each is in the helper,
`cc.SUPPORTED`), marked as an assumption. These fix the ROI width (w1, which the claims file
calls a standard choice), the ROI on the centre (O3 a, an assumption: Methods 2.4 measures
"from the geometrical center"), per-dish normalization (O5 a, the claims file), raw intensity
(O6 a, Methods 2.4 names no background for the monolayers) and per-dish ROI means (O7 a). Both
diagonals averaged (O2 a) is fixed for Fig. 5 only (`cc.SUPPORTED_FIG5`), whose legend says so;
for Figs. 6 to 9 the claims file says the paper does not say, and O2 is varied. The ROI count
(O1) and the end ROI (O4) have no basis and are always varied. Where the supported readings still disagree, no reading can be adopted
and the claim is blocked as `methods_underspecified`, with every result shown (`decide`
returns None).

O8 (paired or unpaired, for comparisons inside the same dishes) is run both ways; the paired
test is the supported reading, an assumption with no basis in the paper: it is the standard test
for two measures taken in the same dish.

In [ ]:
def dishes(ds, **cond):
    return [r for r in DATA[ds] if all(r["cond"][k] == v for k, v in cond.items())]


def ttest(a, b, paired=False, welch=False):
    a, b = np.asarray(a, float), np.asarray(b, float)
    if paired:
        return stats.ttest_rel(a, b)
    return stats.ttest_ind(a, b, equal_var=not welch)


def readings_table(fn, **fixed):
    """Apply fn(reading) -> dict to every combination of readings; return a DataFrame."""
    out = []
    for rd in cc.iter_readings(**fixed):
        res = fn(rd)
        if res is not None:
            out.append({**rd, **res})
    return pd.DataFrame(out)


def show_spread(df, cols, by=("o1", "w", "o2", "o3", "o4", "o5", "o6", "o7")):
    """Range of each value over the readings, per value of each reading."""
    keep = [b for b in by if b in df and df[b].nunique() > 1]
    for b in keep:
        print(f"by {b}:")
        print(df.groupby(b)[cols].agg(["min", "max"]).round(3).to_string())


def primary(df):
    """The main reading (the first reading of each option), used to report numbers."""
    m = np.ones(len(df), bool)
    for k, v in cc.PRIMARY.items():
        if k in df:
            m &= (df[k] == v).values
    return df[m].iloc[0]


def supported(df, sup=None):
    """The readings used to decide (cc.SUPPORTED, or cc.SUPPORTED_FIG5 for Fig. 5); O1 and O4 vary,
    and O2 too outside Fig. 5."""
    m = np.ones(len(df), bool)
    for k, v in (sup or cc.SUPPORTED).items():
        if k in df and df[k].nunique() > 1:
            m &= (df[k] == v).values
    return df[m]


def decide(df, col, label=None, sup=None):
    """Outcome of a test over readings, by the claims file's rule: the one all readings give;
    where they differ, the one all the supported readings give (an assumption); else None."""
    share = float(df[col].astype(bool).mean())
    sup = supported(df, sup)[col].astype(bool)
    if share in (0.0, 1.0):
        out, assumed = share == 1.0, False
    elif sup.mean() in (0.0, 1.0):
        out, assumed = bool(sup.mean() == 1.0), True
    else:
        out, assumed = None, True
    print(f"{label or col}: holds in {share:.0%} of {len(df)} readings, in {sup.mean():.0%} of the "
          f"{len(sup)} supported readings -> {out}{' (assumed)' if assumed and out is not None else ''}")
    return out, assumed, share

## Blocked claims: Figures 3 and 4

fig3-field-simulation, fig4-sham-no-uptake, fig4-unipolar-strongest, fig4-bipolar-weaker,
fig4-twenty-fold-cancellation and fig8-cancellation-predicts-targeting rest on the single-cell
time lapse of Fig. 4 (Methods 2.2) or the Sim4Life field maps of Fig. 3. The cell below lists
what every file of the five dandisets holds: one or two still images of a monolayer, with no
time series of single cells and no field map.

In [ ]:
print(inv.groupby("dandiset").agg(files=("file", "size"), frames=("frames", "sum"),
                                  sizes=("size", lambda s: sorted(set(s)))).to_string())
print("series names:", sorted({s for r in ALL for s in r["series"]}))

## fig5a-trains-stronger

> Without rotation, 40 unipolar trains from e1 cause much stronger electroporation throughout
> the array than 40 Packet 2 packets started from e1, and the uptake after trains follows the
> field strength distribution.

Profile along the e1 to e3 diagonal (Fig. 5A legend). The files do not say which needle is
e1. With trains only e1 is energized, so e1 is the needle whose outer quarter of the half-diagonal
is brightest in the train dishes (printed below); e1 is taken as the needle at that image
corner in both groups, and e3 as the opposite one.

In [ ]:
G5A = {"40P": dishes("000686", protocol="40P"), "40T": dishes("000686", protocol="40T")}
for k, v in G5A.items():
    for r in v:
        p = r["prof"][("a", "w1", "a")]
        q = len(p["d"]) // 4
        ends = {h: round(float(np.nanmean(p["halves"][h][-q:]))) for h in p["halves"]}
        print(k, r["subject_id"], "outer quarter of each half-diagonal, mean:", ends, "brightest:", max(ends, key=ends.get))

E1, E3 = "TR", "BL"


def e1e3_values(group, rd):
    """e1 end, centre, e3 end and the trough on the e1 side, per dish (one diagonal)."""
    out = dict(e1=[], c=[], e3=[], prof_e1=[], prof_e3=[])
    for r in group:
        p = r["prof"][(rd["o1"], rd["w"], rd["o6"])]
        c = cc.centre_value(p, [E1, E3], rd["o3"])
        out["e1"].append(p["halves"][E1][-1])
        out["e3"].append(p["halves"][E3][-1])
        out["c"].append(c)
        out["prof_e1"].append(np.r_[c, p["halves"][E1]])
        out["prof_e3"].append(np.r_[c, p["halves"][E3]])
    out = {k: np.array(v) for k, v in out.items()}
    out["d"] = np.r_[0, G5A["40P"][0]["prof"][(rd["o1"], rd["w"], rd["o6"])]["d"]]
    return out


def fig5a_trains(rd):
    P, T = e1e3_values(G5A["40P"], rd), e1e3_values(G5A["40T"], rd)
    res = {}
    for place in ("e1", "c", "e3"):
        t = ttest(T[place], P[place])
        res[f"T/P {place}"] = T[place].mean() / P[place].mean()
        res[f"p {place}"] = t.pvalue
        res[f"T>P {place}"] = (T[place].mean() > P[place].mean()) and t.pvalue < 0.05
    for pr in ("paired", "unpaired"):
        t = ttest(T["e1"], T["c"], paired=pr == "paired")
        res[f"p T e1>c {pr}"] = t.pvalue
        res[f"T e1>c {pr}"] = (T["e1"].mean() > T["c"].mean()) and t.pvalue < 0.05
    return res


t5a = readings_table(fig5a_trains, o2="a", o4="a", o5="a", o7="raw")
print(len(t5a), "readings (O1, width, O3, O6, O7 raw for per-dish tests)")
cols = [c for c in t5a.columns if c.startswith(("T/P", "p "))]
print(t5a[["o1", "w", "o3", "o6"] + cols].round(4).to_string())

In [ ]:
rdp = dict(cc.PRIMARY)
P5, T5 = e1e3_values(G5A["40P"], rdp), e1e3_values(G5A["40T"], rdp)
for name, X in (("40 packets", P5), ("40 trains", T5)):
    print(f"{name} (n = {len(X['c'])}): e1 end {X['e1'].mean():.0f} +/- {stats.sem(X['e1']):.0f}, "
          f"centre {X['c'].mean():.0f} +/- {stats.sem(X['c']):.0f}, e3 end {X['e3'].mean():.0f} "
          f"+/- {stats.sem(X['e3']):.0f} a.u. (mean +/- s.e.)")
for place in ("e1", "c", "e3"):
    t = ttest(T5[place], P5[place]); tw = ttest(T5[place], P5[place], welch=True)
    print(f"trains vs packets at {place}: ratio {T5[place].mean() / P5[place].mean():.2f}, "
          f"Student p = {t.pvalue:.2g}, Welch p = {tw.pvalue:.2g}")
for pr in ("paired", "unpaired"):
    t = ttest(T5["e1"], T5["c"], paired=pr == "paired")
    print(f"trains, e1 end vs centre ({pr}): {T5['e1'].mean():.0f} vs {T5['c'].mean():.0f}, p = {t.pvalue:.2g}")
need = ["T>P e1", "T>P c", "T>P e3"]
t5a["holds unpaired"] = t5a[need].all(axis=1) & t5a["T e1>c unpaired"]
t5a["holds paired"] = t5a[need].all(axis=1) & t5a["T e1>c paired"]
for c in need + ["T e1>c paired", "T e1>c unpaired", "holds unpaired"]:
    print(f"{c}: holds in {t5a[c].mean():.0%} of {len(t5a)} readings")
OUT5A = decide(t5a, "holds paired", "fig5a-trains-stronger (O8 paired)")
fig, ax = plt.subplots(1, 2, figsize=(11, 3.5))
for X, name in ((P5, "40 packets"), (T5, "40 trains")):
    d = X["d"]
    x = np.r_[-d[::-1], d[1:]]
    y = np.r_[X["prof_e1"].mean(0)[::-1], X["prof_e3"].mean(0)[1:]]
    ax[0].plot(x, y, label=name)
    ax[1].plot(x, y / y[0] * 100, label=name)
ax[0].set_xlabel("distance from centre along e1 (-) to e3 (+), mm"); ax[0].set_ylabel("YP, a.u.")
ax[1].set_xlabel("same"); ax[1].set_ylabel("% of e1 end ROI"); ax[0].legend()
plt.tight_layout(); plt.show()
print(f"plotted ranges: packets {np.nanmin(P5['prof_e1']):.0f} to {np.nanmax(P5['prof_e1']):.0f}, "
      f"trains {np.nanmin(T5['prof_e1']):.0f} to {np.nanmax(T5['prof_e3']):.0f} a.u. (dish values)")

In [ ]:
# Coverage of the end ROIs. YO-PRO-1 marks only electroporated cells, so a dark ROI may be
# unaffected cells or bare glass. For each end ROI of the main reading: its mean, its mean over the
# dish's cell-free level (the O6 b background, 1st percentile inside the electrode square), the
# share of its pixels at or below that level, and the share outside the image (value 0). The same
# for a 0.5-mm square around the ROI, to see the surroundings.
cov = []
for k, v in G5A.items():
    for r in v:
        fr, P = r["frames"][0], r["geom"]["P"]
        names, c = cc.corner_names(P), P.mean(0)
        p = r["prof"][("a", "w1", "a")]
        bg, d_end = p["background"], p["d"][-1] * 1000 / cc.UM_PER_PX
        for end, h in (("e1", E1), ("e3", E3)):
            u = (names[h] - c) / np.linalg.norm(names[h] - c)
            for size, w_px in (("ROI", p["s"] * 1000 / cc.UM_PER_PX), ("0.5 mm", 500 / cc.UM_PER_PX)):
                vals = cc.roi_samples(fr, c, u, d_end, w_px, min(w_px / 4, 1.0))
                cov.append(dict(group=k, dish=r["subject_id"].split("(")[-1].rstrip(")"), end=end, area=size,
                                mean=vals[vals > 0].mean(), background=bg,
                                over_bg=vals[vals > 0].mean() / bg, at_or_below_bg=(vals <= bg).mean(),
                                outside_image=(vals <= 0).mean()))
cov = pd.DataFrame(cov)
print(cov.round(3).to_string())
roi = cov[cov.area == "ROI"]
print("\nend ROI over the cell-free level, range by group and end:")
print(roi.groupby(["group", "end"])[["over_bg", "at_or_below_bg", "outside_image"]].agg(["min", "max"]).round(3).to_string())
# Crops of 2 mm around each end ROI (e1 top row, e3 bottom row), the ROI in red.
fig, axs = plt.subplots(2, 8, figsize=(20, 6.2), constrained_layout=True)
for j, r in enumerate(G5A["40P"] + G5A["40T"]):
    fr, P = r["frames"][0], r["geom"]["P"]
    names, c = cc.corner_names(P), P.mean(0)
    p = r["prof"][("a", "w1", "a")]
    half = int(1000 / cc.UM_PER_PX)
    for i, h in enumerate((E1, E3)):
        u = (names[h] - c) / np.linalg.norm(names[h] - c)
        e = c + u * p["d"][-1] * 1000 / cc.UM_PER_PX
        x0, y0 = int(e[0]) - half, int(e[1]) - half
        crop = fr[max(y0, 0):y0 + 2 * half, max(x0, 0):x0 + 2 * half]
        ax = axs[i, j]
        ax.imshow(crop, cmap="gray", vmin=0, vmax=np.percentile(fr[fr > 0], 99.5))
        w_px = p["s"] * 1000 / cc.UM_PER_PX
        ax.add_patch(plt.Rectangle((e[0] - max(x0, 0) - w_px / 2, e[1] - max(y0, 0) - w_px / 2), w_px, w_px,
                                   fill=False, color="r", lw=1.5))
        ax.set_title(f"{r['cond']['protocol']} {r['subject_id'].split('(')[-1].rstrip(')')} {'e1' if i == 0 else 'e3'}", fontsize=8)
        ax.axis("off")
plt.suptitle("2 x 2 mm around the e1 (top) and e3 (bottom) end ROIs, same grey scale within a dish (0 to its 99.5th percentile)")
plt.show()

**Result: not reproduced.** In all 16 readings trains exceed packets at the e1 end (3.03 times in
the main reading, p = 0.0093) and in the centre (2.02 times, p = 0.0034), and the train profile
falls from the e1 end to the centre (18094 against 7086 a.u., paired p = 0.0076, unpaired
p = 0.0015), as the field does with distance from e1. At the e3 end the trains are not stronger
than the packets in any reading (0.75 times, p = 0.40). The criterion asks for trains above
packets at all three places and names no partial outcome, so the claim is not reproduced. The
e3 end ROIs lie inside the image (no pixel outside it) and are 2.0 to 4.2 times the dish's
cell-free level in the train dishes and 2.2 to 5.0 times in the packet dishes, with no pixel at
or below that level, so the e3 values are not read from off-image pixels. YO-PRO-1 marks only
electroporated cells, so the images cannot tell unaffected cells from bare glass.

## fig5a-packet-center-enhanced

> Without rotation, Packet 2 enhances electroporation in the center of the array despite the
> weakening of the field with distance from e1, but uptake near e1 stays stronger than in the
> center.

Reading (a): the packet centre above the trough between the centre and the e1 end (O9, on the
e1 half, chosen on the mean profile; paired and unpaired, O8). Reading (b): the per-dish ratio
centre / e1 end higher for packets than trains (unpaired). Second part: the e1 end above the
centre for packets (paired and unpaired).

In [ ]:
def fig5a_packets(rd):
    P, T = e1e3_values(G5A["40P"], rd), e1e3_values(G5A["40T"], rd)
    mp = P["prof_e1"].mean(0)
    ti = 1 + int(np.nanargmin(mp[1:-1]))
    tr = P["prof_e1"][:, ti]
    res = dict(trough_mm=P["d"][ti], centre=P["c"].mean(), trough=tr.mean(), e1=P["e1"].mean())
    for pr in ("paired", "unpaired"):
        t = ttest(P["c"], tr, paired=pr == "paired")
        res[f"a {pr}"] = (P["c"].mean() > tr.mean()) and t.pvalue < 0.05
        res[f"p a {pr}"] = t.pvalue
        t = ttest(P["e1"], P["c"], paired=pr == "paired")
        res[f"e1>c {pr}"] = (P["e1"].mean() > P["c"].mean()) and t.pvalue < 0.05
        res[f"p e1>c {pr}"] = t.pvalue
    rp, rt = P["c"] / P["e1"], T["c"] / T["e1"]
    t = ttest(rp, rt)
    res.update({"ratio P": rp.mean(), "ratio T": rt.mean(), "p b": t.pvalue,
                "b": (rp.mean() > rt.mean()) and t.pvalue < 0.05})
    return res


t5p = readings_table(fig5a_packets, o2="a", o4="a", o5="a", o7="raw")
print(t5p[["o1", "w", "o3", "o6", "trough_mm", "centre", "trough", "e1", "p a paired", "p a unpaired",
           "ratio P", "ratio T", "p b", "p e1>c paired", "p e1>c unpaired"]].round(4).to_string())
for pr in ("paired", "unpaired"):
    t5p[f"enh {pr}"] = t5p[f"a {pr}"] | t5p["b"]
for c in ["a paired", "a unpaired", "b", "enh paired", "enh unpaired", "e1>c paired", "e1>c unpaired"]:
    print(f"{c}: holds in {t5p[c].mean():.0%} of {len(t5p)} readings")
pm = primary(t5p)
print(f"main reading: centre {pm['centre']:.0f}, trough {pm['trough']:.0f} at {pm['trough_mm']:.2f} mm, "
      f"e1 end {pm['e1']:.0f} a.u.; p(a) paired {pm['p a paired']:.3g} unpaired {pm['p a unpaired']:.3g}; "
      f"centre/e1 packets {pm['ratio P']:.2f} vs trains {pm['ratio T']:.2f}, p = {pm['p b']:.3g}; "
      f"e1 > centre p paired {pm['p e1>c paired']:.3g} unpaired {pm['p e1>c unpaired']:.3g}")
for r_, c_, e_ in zip(G5A["40P"], P5["c"], P5["e1"]):
    print("  ", r_["subject_id"], f"centre {c_:.0f}  e1 end {e_:.0f}")
OUT5P_ENH = decide(t5p, "enh paired", "enhanced in the centre, (a) or (b), O8 paired")
OUT5P_E1 = decide(t5p, "e1>c paired", "e1 end above the centre, O8 paired")

**Result: partially reproduced.** Reading (a) of the enhancement holds by the paired test in all
16 readings: the packet centre is above the trough on the e1 side (3512 against 1326 a.u. at
1.03 mm; p = 0.016 paired, 0.017 unpaired). Reading (b) does not (centre over e1 end 0.89 for
packets against 0.40 for trains, p = 0.10). The e1 end is not significantly above the centre in
any reading (5968 against 3512 a.u., p = 0.47 paired and 0.39 unpaired), and in 2 of the 4
dishes the centre is the brighter of the two. One part of the claim holds, the criterion's
partial outcome.

## fig5b-rotation-center-peak

> With rotation (1P x 40), Packet 2 smears the effect at the perimeter and brings out a remote
> effect in the center, while the same rotation of unipolar trains (1T x 40) leaves a void of
> electroporation in the center.

Both diagonals averaged (Fig. 5B legend). 1P x 40: centre above its trough (O9 a and b), paired
and unpaired. 1T x 40: void read as (a) centre below its most peripheral value by the same test,
or (b) the centre ROI is the lowest point of the mean profile.

In [ ]:
G5B = {"1Px40": dishes("000686", protocol="1Px40"), "1Tx40": dishes("000686", protocol="1Tx40")}


def fig5b(rd):
    gp = cc.group_values(G5B["1Px40"], rd)
    gt = cc.group_values(G5B["1Tx40"], rd)
    res = dict(P_centre=gp["centre"], P_trough=gp["trough_a"], P_trough_mm=gp["trough_d"],
               T_centre=gt["centre"], T_end=gt["end"], T_min_mm=gt["d"][int(np.nanargmin(gt["profile"]))])
    for pr in ("paired", "unpaired"):
        t = ttest(gp["dish_centre"], gp["dish_trough"], paired=pr == "paired")
        res[f"p P c>tr {pr}"] = t.pvalue
        res[f"P c>tr {pr}"] = (gp["centre"] > gp["trough_a"]) and t.pvalue < 0.05
        t = ttest(gt["dish_centre"], gt["dish_end"], paired=pr == "paired")
        res[f"p T c<end {pr}"] = t.pvalue
        res[f"void a {pr}"] = (gt["centre"] < gt["end"]) and t.pvalue < 0.05
    res["void b"] = int(np.nanargmin(gt["profile"])) == 0
    return res


t5b = readings_table(fig5b, o4="a", o5="a", o7="raw")
show_spread(t5b, ["P_centre", "P_trough", "p P c>tr paired", "p P c>tr unpaired", "T_centre", "T_end",
                  "p T c<end paired", "p T c<end unpaired", "T_min_mm"])
for pr in ("paired", "unpaired"):
    t5b[f"holds {pr}"] = t5b[f"P c>tr {pr}"] & (t5b[f"void a {pr}"] | t5b["void b"])
for c in ["P c>tr paired", "P c>tr unpaired", "void a paired", "void a unpaired", "void b",
          "holds paired", "holds unpaired"]:
    print(f"{c}: holds in {t5b[c].mean():.0%} of {len(t5b)} readings")
OUT5B = decide(t5b, "holds paired", "fig5b-rotation-center-peak (O8 paired)", sup=cc.SUPPORTED_FIG5)
print("main reading, paired:", bool(primary(t5b)["holds paired"]), "; unpaired:", bool(primary(t5b)["holds unpaired"]))
print(supported(t5b, cc.SUPPORTED_FIG5)[["o1", "w", "o2", "o3", "o6", "P_centre", "P_trough", "p P c>tr paired", "T_centre", "T_end",
                      "p T c<end paired", "T_min_mm", "holds paired"]].round(4).to_string())
pm = primary(t5b)
print(pm[["P_centre", "P_trough", "P_trough_mm", "p P c>tr paired", "p P c>tr unpaired", "T_centre", "T_end",
          "p T c<end paired", "p T c<end unpaired", "T_min_mm"]].to_string())
gp = cc.group_values(G5B["1Px40"], cc.PRIMARY); gt = cc.group_values(G5B["1Tx40"], cc.PRIMARY)
fig, ax = plt.subplots(1, 2, figsize=(11, 3.5))
for gg, name in ((gp, "1P x 40"), (gt, "1T x 40")):
    ax[0].plot(gg["d"], gg["profile"], label=name); ax[1].plot(gg["d"], gg["norm_profile"], label=name)
ax[0].set_ylabel("YP, a.u."); ax[1].set_ylabel("% of most peripheral ROI")
for a in ax: a.set_xlabel("distance from centre, mm (both diagonals)")
ax[0].legend(); plt.tight_layout(); plt.show()
print(f"ranges: 1P x 40 {gp['profile'].min():.0f} to {gp['profile'].max():.0f}, "
      f"1T x 40 {gt['profile'].min():.0f} to {gt['profile'].max():.0f} a.u.")

**Result: blocked, methods_underspecified.** With the paired test, the supported reading of O8,
the 1P x 40 centre is above its trough in 96% of the 48 readings (main reading 2526 against
1050 a.u. at 2.13 mm, paired p = 0.016). The 1T x 40 void, read as the centre below the most
peripheral ROI (5376 against 11473 a.u.), holds by the paired test in 62% of readings and fails
in the main one (p = 0.055); the centre ROI is the lowest point of the 1T x 40 profile in 8% of
readings. The two supported readings, which differ only in the ROI count (O1), disagree: the
claim fails with 92 ROIs per diagonal (void p = 0.055) and holds with the count split over the
two diagonals (void p = 0.020). The paper gives no basis to choose between them, so the claim is
blocked. With the unpaired test the claim holds in 98% of readings, including the main one.

## fig5c-normalized-contrast

> Normalized to the most peripheral ROI (100%), rotated Packet 2 and rotated trains show sharply
> different spatial distributions, the packet's center being relatively high and the trains'
> relatively low.

Per-dish normalized centres (O5 a), 1P x 40 against 1T x 40, unpaired t-test.

In [ ]:
def fig5c(rd):
    gp = cc.group_values(G5B["1Px40"], rd); gt = cc.group_values(G5B["1Tx40"], rd)
    t = ttest(gp["dish_norm_centre"], gt["dish_norm_centre"])
    return dict(P_norm=gp["dish_norm_centre"].mean(), T_norm=gt["dish_norm_centre"].mean(), p=t.pvalue,
                p_welch=ttest(gp["dish_norm_centre"], gt["dish_norm_centre"], welch=True).pvalue,
                holds=(gp["dish_norm_centre"].mean() > gt["dish_norm_centre"].mean()) and t.pvalue < 0.05)


t5c = readings_table(fig5c, o2="a", o5="a", o7="raw")
show_spread(t5c, ["P_norm", "T_norm", "p"])
print(f"holds in {t5c['holds'].mean():.0%} of {len(t5c)} readings; 1P x 40 above 100% in "
      f"{(t5c['P_norm'] > 100).mean():.0%}")
pm = primary(t5c)
print(f"main reading: 1P x 40 {pm['P_norm']:.0f}%, 1T x 40 {pm['T_norm']:.0f}%, Student p = {pm['p']:.2g}, "
      f"Welch p = {pm['p_welch']:.2g}")
print("per dish 1P x 40:", np.round(gp["dish_norm_centre"]).tolist(), " 1T x 40:", np.round(gt["dish_norm_centre"]).tolist())

**Result: blocked, because it depends on fig5b-rotation-center-peak.** Its own test holds: the
normalized centre is 213% for 1P x 40 and 57% for 1T x 40 (Student p = 0.0088, Welch p = 0.018),
the difference is significant in all 32 readings, and the 1P x 40 centre is above 100% in all of
them.

## fig6a-more-packets-more-uptake

> Raising the total number of rotated Packet 2 packets from 4 (1Px4) to 20 (5Px4) and 40
> (10Px4) strongly raises YP uptake throughout the array, and a central peak is discernible for
> every treatment.

000875 (3.2 kV, small array, read from the files: O11). At the most peripheral ROI, the trough
and the centre: means rise 4 < 20 < 40 and 40 > 4 by an unpaired t-test; for each treatment the
centre above the trough (paired and unpaired, O8).

In [ ]:
G6 = {k: dishes("000875", protocol=k) for k in ("1Px4", "5Px4", "10Px4")}
print({k: len(v) for k, v in G6.items()}, "dishes; voltages", sorted({r["cond"]["voltage_kv"] for r in DATA["000875"]}),
      "; array", sorted({r["cond"]["array"] for r in DATA["000875"]}))


def fig6a(rd):
    gv = {k: cc.group_values(v, rd) for k, v in G6.items()}
    res = {}
    rise_all = True
    for place, key in (("end", "dish_end"), ("trough", "dish_trough"), ("centre", "dish_centre")):
        m = [gv[k][key].mean() for k in G6]
        t = ttest(gv["10Px4"][key], gv["1Px4"][key])
        ok = m[0] < m[1] < m[2] and t.pvalue < 0.05
        res[f"{place} 1/5/10"] = "/".join(f"{x:.0f}" for x in m)
        res[f"p {place} 10 vs 1"] = t.pvalue
        res[f"rise {place}"] = ok
    for k in G6:
        for pr in ("paired", "unpaired"):
            t = ttest(gv[k]["dish_centre"], gv[k]["dish_trough"], paired=pr == "paired")
            res[f"peak {k} {pr}"] = (gv[k]["dish_centre"].mean() > gv[k]["dish_trough"].mean()) and t.pvalue < 0.05
            res[f"p peak {k} {pr}"] = t.pvalue
    return res


t6a = readings_table(fig6a, o4="a", o5="a", o7="raw")
pk = [c for c in t6a.columns if c.startswith("peak ")]
t6a["rise_n"] = t6a[["rise end", "rise trough", "rise centre"]].sum(axis=1)
t6a["peaks unpaired"] = t6a[[c for c in pk if c.endswith("unpaired")]].all(axis=1)
t6a["peaks paired"] = t6a[[c for c in pk if c.endswith(" paired")]].all(axis=1)
print(t6a[["o1", "w", "o2", "o3", "o6", "end 1/5/10", "trough 1/5/10", "centre 1/5/10", "p end 10 vs 1",
           "p trough 10 vs 1", "p centre 10 vs 1", "rise_n", "peaks unpaired"]].round(4).to_string())
for c in ["rise end", "rise trough", "rise centre", "peaks unpaired", "peaks paired"] + pk:
    print(f"{c}: holds in {t6a[c].mean():.0%} of {len(t6a)} readings")
pm = primary(t6a)
print(pm[[c for c in t6a.columns if c.startswith("p ")]].to_string())
OUT6A_PEAK = decide(t6a, "peaks paired", "centre above trough for every treatment, O8 paired")
print(supported(t6a)[["o1", "w", "o2", "o3", "o6"] + [c for c in t6a.columns if c.startswith("p peak") and c.endswith(" paired")]].round(4).to_string())

**Result: partially reproduced.** In all 48 readings uptake rises from 4 to 20 to 40 packets at
the most peripheral ROI (558, 3515 and 9472 a.u. in the main reading), the trough (458, 1189,
1567) and the centre (726, 2366, 5800), and 40 packets exceed 4 at each place (p = 0.0027, 0.0014
and 0.00015). The centre is above the trough for every treatment in none of the readings, paired
or unpaired: by the paired test it holds for 10Px4 in 83% of readings (main p = 0.0024) but for
1Px4 in none (main p = 0.20) and for 5Px4 in 8% (main p = 0.13). The criterion's partial outcome
names a rise at one or two places; here the rise holds at all three and the peak test fails for
two treatments, which is read as partial, an assumption (asked of a reader in `questions/pakhomov-2023.yaml`).
With O2 varied (one or both diagonals) the peak test still fails for every treatment in all six
supported readings.

## fig6b-peak-relative-to-periphery

> The central peak is almost twofold brighter than the periphery for 1Px4, and about the same
> as the periphery for 5Px4 and 10Px4.

Normalized centre (O1 to O7). Expected 200 +/- 20, 100 +/- 10 and 100 +/- 10 %.

In [ ]:
def fig6b(rd):
    gv = {k: cc.group_values(v, rd) for k, v in G6.items()}
    n = {k: gv[k]["norm_centre"] for k in G6}
    return {"1Px4": n["1Px4"], "5Px4": n["5Px4"], "10Px4": n["10Px4"],
            "match": 180 <= n["1Px4"] <= 220 and 90 <= n["5Px4"] <= 110 and 90 <= n["10Px4"] <= 110,
            "partial": 180 <= n["1Px4"] <= 220 and n["1Px4"] > max(n["5Px4"], n["10Px4"])}


t6b = readings_table(fig6b)
show_spread(t6b, ["1Px4", "5Px4", "10Px4"])
print(f"\n{len(t6b)} readings; all three within tolerance in {t6b['match'].sum()}; 1Px4 within tolerance "
      f"and above both others in {t6b['partial'].sum()}")
print("ranges over readings:", {k: (round(t6b[k].min()), round(t6b[k].max())) for k in ["1Px4", "5Px4", "10Px4"]})
print("1Px4 above both others in", f"{(t6b['1Px4'] > t6b[['5Px4', '10Px4']].max(axis=1)).mean():.0%}", "of readings")
print("main reading:", primary(t6b)[["1Px4", "5Px4", "10Px4"]].round(1).to_dict())
print("readings where 1Px4 is within tolerance and above both others:")
print(t6b[t6b.partial][list(cc.READINGS) + ["1Px4", "5Px4", "10Px4"]].round(1).to_string())
print("closest reading to (200, 100, 100):")
dist = ((t6b["1Px4"] - 200) / 20) ** 2 + ((t6b["5Px4"] - 100) / 10) ** 2 + ((t6b["10Px4"] - 100) / 10) ** 2
print(t6b.loc[dist.idxmin()].to_string())

**Result: partially reproduced.** No reading puts all three values within tolerance. The main
reading gives 128% (1Px4), 70% (5Px4) and 64% (10Px4). In 6 of 576 readings, all with background
subtracted, raw values, w1 and the d1 diagonal, the 1Px4 value is within tolerance (183 to 186%)
and above both others, the criterion's partial outcome. 1Px4 is the highest of the three in 99%
of readings. Over readings the values span 105 to 186%, 35 to 170% and 27 to 110%.

## fig6-trough-half-of-center

> For all three packet numbers, the least affected area between the center and the electrodes
> shows about 2-fold lower fluorescence than the center.

Centre over trough (O9 a and b), expected 2 +/- 0.2 for each treatment.

In [ ]:
rows6 = []
for rd in cc.iter_readings(o4="a", o5="a"):
    for o9 in ("a", "b"):
        v = {k: cc.group_values(g_, rd)[f"centre_over_trough_{o9}"] for k, g_ in G6.items()}
        rows6.append({**rd, "o9": o9, **v})
t6t = pd.DataFrame(rows6)
t6t["n_in"] = ((t6t[list(G6)] >= 1.8) & (t6t[list(G6)] <= 2.2)).sum(axis=1)
t6t["all_above_1"] = (t6t[list(G6)] > 1).all(axis=1)
show_spread(t6t, list(G6), by=("o1", "w", "o2", "o3", "o6", "o7", "o9"))
print(f"\n{len(t6t)} readings; all three in 1.8 to 2.2 in {(t6t.n_in == 3).sum()}; one or two in range and all "
      f"above 1 in {((t6t.n_in > 0) & (t6t.n_in < 3) & t6t.all_above_1).sum()}")
print("main reading (O9 a):", primary(t6t[t6t.o9 == "a"])[list(G6)].round(2).to_dict())
print("main reading (O9 b):", primary(t6t[t6t.o9 == "b"])[list(G6)].round(2).to_dict())
print("ranges:", {k: (round(t6t[k].min(), 2), round(t6t[k].max(), 2)) for k in G6})

**Result: partially reproduced.** No reading puts all three centre-over-trough ratios in 1.8 to
2.2. In the main reading they are 1.59 (1Px4), 1.99 (5Px4) and 3.70 (10Px4); 114 of 288 readings
have one or two in range with all above 1. The ratio grows with the number of packets in the main
reading, where the paper reports about 2 for all three.

## fig6-peak-narrower

> The central peak becomes narrower with more packets.

Widths on the mean normalized profile: (a) full width at half height above the trough;
(b) distance between the two troughs; each also on the LOESS fit (O7).

In [ ]:
roww = []
for rd in cc.iter_readings(o4="a", o5="a"):
    w = {}
    for k, g_ in G6.items():
        gv = cc.group_values(g_, rd)
        fw, tt = cc.peak_width(gv["d"], gv["norm_profile"])
        w[k] = (fw, tt)
    for wi, wname in ((0, "fwhm"), (1, "troughs")):
        x = [w[k][wi] for k in G6]
        roww.append({**rd, "width": wname, "1Px4": x[0], "5Px4": x[1], "10Px4": x[2],
                     "falls": x[0] > x[1] > x[2], "partial": x[2] < x[0] and not (x[0] > x[1] > x[2])})
t6w = pd.DataFrame(roww)
show_spread(t6w, ["1Px4", "5Px4", "10Px4"], by=("width", "o1", "w", "o2", "o6", "o7"))
for wname in ("fwhm", "troughs"):
    s = t6w[t6w.width == wname]
    print(f"{wname}: falls 1 > 5 > 10 in {s.falls.mean():.0%}, 10 narrower than 1 only in {s.partial.mean():.0%} "
          f"of {len(s)} readings; main reading", primary(s)[["1Px4", "5Px4", "10Px4"]].round(3).to_dict(), "mm")
print(f"overall: falls in {t6w.falls.mean():.0%}, partial in {t6w.partial.mean():.0%}")
fig, ax = plt.subplots(figsize=(6, 3.5))
for k, g_ in G6.items():
    gv = cc.group_values(g_, cc.PRIMARY)
    ax.plot(gv["d"], gv["norm_profile"], label=k)
ax.set_xlabel("distance from centre, mm"); ax.set_ylabel("% of most peripheral ROI"); ax.legend()
plt.tight_layout(); plt.show()
OUT6W = decide(t6w[t6w.width == "fwhm"], "falls", "width at half height falls 1 > 5 > 10")
decide(t6w[t6w.width == "troughs"], "falls", "distance between troughs falls 1 > 5 > 10")
print(supported(t6w)[["width", "o1", "o2", "1Px4", "5Px4", "10Px4", "falls"]].round(3).to_string())
print("share of readings where the width falls 1 > 5 > 10, by diagonal (O2):")
print(t6w.groupby(["width", "o2"])["falls"].mean().round(2).unstack().to_string())

**Result: blocked, methods_underspecified (the supported readings disagree).** The full width
at half height above the trough falls from 0.657 mm (1Px4) to 0.636 mm (5Px4) to 0.438 mm
(10Px4) in the main reading and in 87% of the 144 readings of this width. The claims file says
the paper does not state the diagonals for Figs. 6 to 9, so O2 is varied in the supported
readings: the width then falls in 4 of the 6 (both diagonals, and d2). On the d1 diagonal alone
5Px4 is wider than 1Px4 under both ROI counts (0.658, 0.804, 0.249 mm with O1 a; 0.697, 0.733,
0.396 mm with O1 b), the criterion's partial outcome. The supported readings thus give both
reproduced and partially reproduced, and the paper gives no basis to choose the diagonal. In all
six, 10Px4 is narrower than 1Px4. The distance between troughs, the criterion's reading (b),
falls in order in 42% of its readings and in 2 of the 6 supported ones. The outcome was
reproduced in rounds 1 and 2, with both diagonals fixed and the width at half height assumed;
the third verification round agreed to the change to blocked.


## fig7a-amplitude-scales-uptake

> With 1Px4 in the small array, raising the first-pulse amplitude from 2.4 to 5.6 kV raises YP
> fluorescence 15.5 times at the most peripheral point, 16.2 times half-way to the center and
> 16.8 times in the center.

In [ ]:
G7S = {v: dishes("001025", array="small", voltage_kv=v) for v in (2.4, 3.2, 4.0, 4.8, 5.6)}
G7L = {v: dishes("001025", array="large", voltage_kv=v) for v in (4.8, 6.4)}
print("small array, 1Px4:", {k: [r["subject_id"] for r in v] for k, v in G7S.items()})
print("large array, 10Px4:", {k: [r["subject_id"] for r in v] for k, v in G7L.items()})
print("protocols:", sorted({(r["cond"]["array"], r["cond"]["protocol"]) for r in DATA["001025"]}))


def fig7a(rd):
    lo, hi = cc.group_values(G7S[2.4], rd), cc.group_values(G7S[5.6], rd)
    r_ = dict(end=hi["end"] / lo["end"], halfway=hi["halfway"] / lo["halfway"], centre=hi["centre"] / lo["centre"])
    r_["n_in"] = sum(abs(r_[k] - e) <= 0.1 * e for k, e in (("end", 15.5), ("halfway", 16.2), ("centre", 16.8)))
    r_["lo_centre"], r_["hi_centre"] = lo["centre"], hi["centre"]
    return r_


t7a = readings_table(fig7a, o5="a")
show_spread(t7a, ["end", "halfway", "centre"], by=("o1", "w", "o2", "o3", "o4", "o6", "o7"))
print(f"\n{len(t7a)} readings; all three ratios within 10% in {(t7a.n_in == 3).sum()}, one or two in "
      f"{((t7a.n_in > 0) & (t7a.n_in < 3)).sum()}")
print("ranges:", {k: (round(t7a[k].min(), 2), round(t7a[k].max(), 2)) for k in ["end", "halfway", "centre"]})
print("readings with one or more ratios within 10%:")
print(t7a[t7a.n_in > 0][list(k for k in cc.READINGS if k != "o5") + ["end", "halfway", "centre", "n_in"]].round(2).to_string())
print("main reading:", primary(t7a)[["end", "halfway", "centre"]].round(2).to_dict())
gv = {v: cc.group_values(g_, cc.PRIMARY) for v, g_ in G7S.items()}
for v in G7S:
    print(f"{v} kV (n = {gv[v]['n']}): end {gv[v]['end']:.0f}, half-way {gv[v]['halfway']:.0f}, centre {gv[v]['centre']:.0f} a.u.; "
          f"background {np.mean([r['prof'][('a','w1','b')]['background'] for r in G7S[v]]):.0f}")

**Result: partially reproduced.** The 5.6 over 2.4 kV ratios are 6.66 at the most peripheral
ROI, 5.71 half-way and 7.33 in the centre in the main reading, less than half the paper's 15.5,
16.2 and 16.8. Over 288 readings they span 5.26 to 13.96, 4.82 to 15.47 and 5.91 to 15.27. No
reading has all three within 10%; 10 readings, all background-subtracted and on the d2 diagonal,
have one within 10%, the criterion's partial outcome. The 2.4 kV group has 4 dishes where the
Fig. 7 legend gives n = 3; all 4 are used.

## fig7b-targeting-independent-of-amplitude

> Normalized to the most peripheral ROI, the profiles for different amplitudes fully overlap;
> amplitude changes the degree of electroporation but not its spatial distribution.

Per-dish normalized values (O5 a) at the centre and the half-way ROI, every pair of voltages,
unpaired two-sided Student's t-test at p < 0.05, no correction. Reproduced when no pair
differs.

In [ ]:
def pairs_test(groups, rd):
    gv = {k: cc.group_values(v, rd) for k, v in groups.items()}
    out = []
    for a, b in itertools.combinations(groups, 2):
        for place in ("centre", "halfway"):
            x, y = gv[a][f"dish_norm_{place}"], gv[b][f"dish_norm_{place}"]
            t = ttest(x, y)
            out.append(dict(a=a, b=b, place=place, diff_pp=y.mean() - x.mean(), p=t.pvalue,
                            p_welch=ttest(x, y, welch=True).pvalue))
    return pd.DataFrame(out), gv


def fig7_absence(groups):
    rows_ = []
    for rd in cc.iter_readings(o5="a", o7="raw"):
        pt, _ = pairs_test(groups, rd)
        rows_.append({**rd, "n_pairs_differ": int((pt.p < 0.05).sum()), "n_tests": len(pt),
                      "min_p": pt.p.min(), "max_abs_diff_pp": pt.diff_pp.abs().max(),
                      "none": bool((pt.p >= 0.05).all())})
    return pd.DataFrame(rows_)


def errorbar_rule(gv, place):
    out = []
    for a, b in itertools.combinations(gv, 2):
        x, y = gv[a][f"dish_norm_{place}"], gv[b][f"dish_norm_{place}"]
        sa, sb = stats.sem(x), stats.sem(y)
        gap = abs(x.mean() - y.mean()) - sa - sb
        out.append(dict(a=a, b=b, place=place, gap=gap, bars=max(sa, sb), differ=gap > max(sa, sb)))
    return pd.DataFrame(out)


def dish_anova(gv):
    """Variant over all ROIs, one value per dish: one-way ANOVA of voltage on each dish's mean
    normalized profile (centre included), and at the centre and half-way."""
    out = {}
    for name, f in (("profile mean", lambda g: np.nanmean(g["dish_norm_profiles"], axis=1)),
                    ("centre", lambda g: g["dish_norm_centre"]), ("half-way", lambda g: g["dish_norm_halfway"])):
        r_ = stats.f_oneway(*[f(g) for g in gv.values()])
        out[name] = (round(float(r_.statistic), 2), float(r_.pvalue))
    return out


def group_sizes(groups, legend_n=3):
    for k, v in groups.items():
        ids = [r["subject_id"] for r in v]
        print(f"  {k} kV: n = {len(v)} {ids}" + ("" if len(v) == legend_n else f"  (legend n = {legend_n})"))

In [ ]:
print("group sizes against the Fig. 7 legend (n = 3):")
group_sizes(G7S)
t7b = fig7_absence(G7S)
show_spread(t7b, ["n_pairs_differ", "min_p", "max_abs_diff_pp"], by=("o1", "w", "o2", "o3", "o4", "o6"))
print("share of readings with no pair differing, by O6:", t7b.groupby("o6")["none"].mean().round(2).to_dict())
OUT7B = decide(t7b, "none", "fig7b: no pair of voltages differs")
print(supported(t7b)[["o1", "w", "o2", "o3", "o4", "o6", "n_pairs_differ", "min_p", "max_abs_diff_pp"]].round(4).to_string())
for o1 in ("a", "b"):
    rd = {**cc.PRIMARY, "o1": o1}
    pt, gv_ = pairs_test(G7S, rd)
    print(f"\nsupported reading O1 ({o1}), every pair:")
    print(pt.round(4).to_string())
    print("error-bar rule:", int(pd.concat([errorbar_rule(gv_, p) for p in ("centre", "halfway")]).differ.sum()),
          "of 20 pairs differ; one-way ANOVA on dish values (F, p):", dish_anova(gv_))
pt, gv = pairs_test(G7S, cc.PRIMARY)
for v in G7S:
    print(f"{v} kV: normalized centre {gv[v]['dish_norm_centre'].mean():.0f}% (dishes {np.round(gv[v]['dish_norm_centre']).tolist()}), "
          f"half-way {gv[v]['dish_norm_halfway'].mean():.0f}% (dishes {np.round(gv[v]['dish_norm_halfway']).tolist()})")
print("\nper-dish normalized values with background subtracted (O6 b, other options as the main reading):")
gvb = {v: cc.group_values(g_, {**cc.PRIMARY, "o6": "b"}) for v, g_ in G7S.items()}
for v in G7S:
    print(f"  {v} kV: centre {np.round(gvb[v]['dish_norm_centre']).tolist()}, half-way {np.round(gvb[v]['dish_norm_halfway']).tolist()}")
# The 2.4 kV group has 4 dishes where the legend gives 3: each left out in turn, over all readings.
print("\n2.4 kV dishes left out one at a time:")
loo = []
for i, r in enumerate(G7S[2.4]):
    g_ = {**G7S, 2.4: [x for j, x in enumerate(G7S[2.4]) if j != i]}
    t_ = fig7_absence(g_)
    loo.append(dict(left_out=r["subject_id"], no_pair_all=t_["none"].mean(), no_pair_supported=supported(t_)["none"].mean(),
                    main_pairs_differ=int(primary(t_)["n_pairs_differ"]), main_min_p=primary(t_)["min_p"]))
print(pd.DataFrame(loo).round(3).to_string())
fig, ax = plt.subplots(1, 2, figsize=(11, 3.5))
for v in G7S:
    ax[0].plot(gv[v]["d"], gv[v]["profile"], label=f"{v} kV")
    ax[1].plot(gv[v]["d"], gv[v]["norm_profile"], label=f"{v} kV")
ax[0].set_ylabel("YP, a.u."); ax[1].set_ylabel("% of most peripheral ROI"); ax[0].legend()
for a in ax: a.set_xlabel("distance from centre, mm")
plt.tight_layout(); plt.show()
print("plotted ranges:", {v: (round(float(np.nanmin(g_["norm_profile"]))), round(float(np.nanmax(g_["norm_profile"])))) for v, g_ in gv.items()}, "%")

**Result: blocked, methods_underspecified.** Over the 96 readings no pair of voltages differs in
27%; the others find one to six pairs. The supported readings disagree: no pair differs in 1 of
the 12 (8%), the main reading (smallest p = 0.052, 2.4 against 4.8 kV at the centre), while with
the ROI count split over the diagonals (O1 b) two pairs differ (2.4 against 4.8 kV, p = 0.026, and
2.4 against 5.6 kV, p = 0.0051, at the centre), and the other supported readings find one to
three. With raw intensity no pair differs in 2% of all readings. The paper gives no basis to
choose the ROI count, the end ROI or the diagonals, so the claim is blocked. Under both O1
readings the error-bar rule of Methods 2.4 finds 3 or 4 of 20 pairs different, and a one-way
ANOVA of the per-dish normalized centres finds a voltage effect (p = 0.028 and 0.021); on each
dish's mean normalized profile it does not (p = 0.26 and 0.12). The 4.8 kV dishes give a
normalized centre of 229% (136, 232 and 318%) against 117 to 129% at the other voltages. Leaving
out any one of the four 2.4 kV dishes leaves the supported readings split (no pair differs in 25%
of them).

## fig7cd-holds-for-large-array

> The independence of the normalized distribution from amplitude also holds when rotating
> multiple packets (10Px4) in the larger array.

Same test on the large array with 10Px4. The voltages are read from the files (O11).

In [ ]:
print("group sizes against the Fig. 7 legend (n = 3):")
group_sizes(G7L)
t7c = fig7_absence(G7L)
show_spread(t7c, ["n_pairs_differ", "min_p", "max_abs_diff_pp"], by=("o1", "w", "o2", "o3", "o4", "o6"))
OUT7C = decide(t7c, "none", "fig7cd: no pair of voltages differs")
print("share of readings with no pair differing, by diagonal (O2):", t7c.groupby("o2")["none"].mean().round(2).to_dict())
print("readings where a pair differs:")
print(t7c[~t7c["none"]][["o1", "w", "o2", "o3", "o4", "o6", "n_pairs_differ", "min_p"]].round(4).to_string())
print(supported(t7c)[["o1", "w", "o2", "o3", "o4", "o6", "n_pairs_differ", "min_p", "max_abs_diff_pp"]].round(4).to_string())
ptl, gvl = pairs_test(G7L, cc.PRIMARY)
print(ptl.round(4).to_string())
for v in G7L:
    g_ = gvl[v]
    print(f"{v} kV (n = {g_['n']}): normalized centre {g_['dish_norm_centre'].mean():.0f}% "
          f"(dishes {np.round(g_['dish_norm_centre']).tolist()}), half-way {g_['dish_norm_halfway'].mean():.0f}% "
          f"(dishes {np.round(g_['dish_norm_halfway']).tolist()}); a.u. end {g_['end']:.0f}, half-way {g_['halfway']:.0f}, centre {g_['centre']:.0f}")
print("rise 4.8 -> 6.4 kV (a.u.): end x%.2f, half-way x%.2f, centre x%.2f" % tuple(
    gvl[6.4][k] / gvl[4.8][k] for k in ("end", "halfway", "centre")))
ebl = pd.concat([errorbar_rule(gvl, p) for p in ("centre", "halfway")])
print("variant, error-bar rule:", int(ebl.differ.sum()), "of", len(ebl), "pairs differ")
# Power of the test at these group sizes: the smallest difference in normalized centre that an
# unpaired t-test with n = 2 and 3 detects with 80% power at p < 0.05, given the pooled s.d.
from statsmodels.stats.power import TTestIndPower
x, y = gvl[4.8]["dish_norm_centre"], gvl[6.4]["dish_norm_centre"]
sd = np.sqrt(((len(x) - 1) * x.var(ddof=1) + (len(y) - 1) * y.var(ddof=1)) / (len(x) + len(y) - 2))
es = TTestIndPower().solve_power(nobs1=len(x), ratio=len(y) / len(x), alpha=0.05, power=0.8)
print(f"pooled s.d. of the normalized centre {sd:.0f} pp; detectable with 80% power: {es * sd:.0f} pp (effect size {es:.2f})")

**Result: blocked, methods_underspecified (the supported readings disagree).**
On the large array with 10Px4, the 4.8 kV (n = 2) and 6.4 kV (n = 3) normalized centres are 204%
and 198% (p = 0.95), and half-way they differ by 53 percentage points (p = 0.14). No pair differs
in 96% of 96 readings. With O2 varied, as the claims file's statement that the paper is silent on Figs. 6
to 9 requires, 11 of the 12 supported readings find none and one, on the d1 diagonal (92 ROIs,
the mean of the end ROIs), finds one pair (p = 0.037); every reading with a pair differing is on
d1. The supported readings disagree, and the paper gives no basis to choose the diagonal, the rule
applied to fig7b-targeting-independent-of-amplitude. The split is thin (1 of 12, p = 0.037). The error-bar rule finds no
difference. The test has little power: with these group sizes and a pooled s.d. of 91 percentage
points, a difference in the normalized centre would need to be about 353 percentage points to be
detected with 80% power. Only two voltages are in the dataset for this array, with 2 dishes at
4.8 kV where the legend gives n = 3. Uptake in a.u. rises with voltage (3.39 times at the most
peripheral ROI, 4.03 times in the centre).
The outcome was reproduced in rounds 1 and 2, with both diagonals fixed; the third verification
round agreed to the change to blocked.


## discussion-independent-of-array-size

> Remote targeting, within the studied limits, depends neither on pulse amplitude nor on array
> size, so findings from small arrays can be extrapolated to larger inter-electrode distances.

Reproduced when the two claims above are both reproduced. The normalized profiles of the two
arrays on distance as a fraction of the measured half-range are shown; the small array received
1Px4 and the large array 10Px4, so packet number and array size are confounded.

In [ ]:
print(f"fig7b: no pair differs in {t7b['none'].mean():.0%} of readings, decided {OUT7B[0]}; "
      f"fig7cd: {t7c['none'].mean():.0%}, decided {OUT7C[0]}")
fig, ax = plt.subplots(figsize=(6, 3.5))
for v, g_ in gv.items():
    ax.plot(g_["d"] / g_["d"][-1], g_["norm_profile"], "b-", alpha=0.3 + 0.12 * list(gv).index(v))
for v, g_ in gvl.items():
    ax.plot(g_["d"] / g_["d"][-1], g_["norm_profile"], "r-", alpha=0.5 + 0.4 * list(gvl).index(v))
ax.set_xlabel("distance / most peripheral ROI distance"); ax.set_ylabel("% of most peripheral ROI")
ax.set_title("blue: small array 1Px4, 2.4 to 5.6 kV; red: large array 10Px4, 4.8 and 6.4 kV")
plt.tight_layout(); plt.show()
print("normalized centre, small array by kV:", {v: round(g_["dish_norm_centre"].mean()) for v, g_ in gv.items()},
      " large array by kV:", {v: round(g_["dish_norm_centre"].mean()) for v, g_ in gvl.items()})

**Result: blocked, because it depends on fig7b-targeting-independent-of-amplitude.**
fig7b is blocked (no pair differs in 27% of its readings), and fig7cd-holds-for-large-array is
blocked too, as its supported readings disagree (no pair differs in 96% of its readings). The normalized
centres are 117 to 229% on the small array (1Px4) and 198 to 204% on the large array (10Px4),
with packet number and array size confounded.

## fig8a-equal-center-uptake

> Packets 1, 2 and 3 at 4 kV, rotated one cycle (1Px4), give the same YP uptake in the center
> of the array, 215 to 235 a.u.

Centre in a.u.; three means inside 193.5 to 258.5 under one reading, and no pair differing by
an unpaired t-test (the test outcome over readings by the rule for tests). The packet of each
dish is read from its session description ("Protocol1/2/3").

In [ ]:
G8 = {k: dishes("001027", packet=k) for k in (1, 2, 3)}
print({k: [r["subject_id"] for r in v] for k, v in G8.items()}, "voltages",
      sorted({r["cond"]["voltage_kv"] for r in DATA["001027"]}), "array", sorted({r["cond"]["array"] for r in DATA["001027"]}))


def fig8(rd, groups=None):
    groups = groups or G8
    gv = {k: cc.group_values(v, rd) for k, v in groups.items()}
    res = {f"centre P{k}": gv[k]["centre"] for k in groups}
    res.update({f"end P{k}": gv[k]["end"] for k in groups})
    res.update({f"norm P{k}": gv[k]["norm_centre"] for k in groups})
    if rd["o7"] == "raw":
        for a, b in itertools.combinations(groups, 2):
            res[f"p centre {a}-{b}"] = ttest(gv[a]["dish_centre"], gv[b]["dish_centre"]).pvalue
        for k in (1, 3):
            t = ttest(gv[k]["dish_end"], gv[2]["dish_end"])
            res[f"p end {k}>2"] = t.pvalue
            res[f"end {k}>2"] = (gv[k]["dish_end"].mean() > gv[2]["dish_end"].mean()) and t.pvalue < 0.01
    return res


t8 = readings_table(fig8)
cent = [f"centre P{k}" for k in G8]
t8["centres_in"] = ((t8[cent] >= 193.5) & (t8[cent] <= 258.5)).all(axis=1)
pc = [c for c in t8.columns if c.startswith("p centre")]
raw8 = t8[t8.o7 == "raw"].copy()
raw8["no_pair_differs"] = (raw8[pc] >= 0.05).all(axis=1)
show_spread(t8, cent, by=("o1", "w", "o2", "o3", "o6", "o7"))
print(f"\nall three centres in 193.5 to 258.5 a.u. in {t8.centres_in.sum()} of {len(t8)} readings")
print(f"no pair differs (raw readings) in {raw8.no_pair_differs.mean():.0%} of {len(raw8)}")
pm = primary(t8)
print("main reading:", pm[cent + pc].round(4).to_dict())
for k in G8:
    print(f"Packet {k}: centre per dish", np.round(cc.group_values(G8[k], cc.PRIMARY)["dish_centre"]).tolist())

**Result: not reproduced.** The centre values are 692 (Packet 1), 378 (Packet 2) and 401 a.u.
(Packet 3) in the main reading, and no reading puts all three in 193.5 to 258.5 a.u. Packet 1
differs from Packet 2 (p = 0.0009) and Packet 3 (p = 0.0074) in the main reading, and some pair
differs in every one of the 192 readings on dish means.

## fig8a-more-uptake-near-electrodes

> Packets 1 and 3 cause much more YP uptake near the electrodes (250 to 350 a.u.) than Packet 2
> (about 120 a.u.), p < 0.01.

Most peripheral ROI; Packets 1 and 3 inside 225 to 385, Packet 2 within 108 to 132 a.u.; each of
Packets 1 and 3 above Packet 2 by an unpaired t-test at p < 0.01.

In [ ]:
t8["ends_in"] = (t8["end P1"].between(225, 385) & t8["end P3"].between(225, 385) & t8["end P2"].between(108, 132))
raw8 = t8[t8.o7 == "raw"].copy()
raw8["tests"] = raw8["end 1>2"].astype(bool) & raw8["end 3>2"].astype(bool)
show_spread(t8, ["end P1", "end P2", "end P3"], by=("o1", "w", "o2", "o4", "o6", "o7"))
print(f"\nvalues in range in {t8.ends_in.sum()} of {len(t8)} readings; both tests at p < 0.01 in "
      f"{raw8.tests.mean():.0%} of {len(raw8)} raw readings; P1 > P2 in {raw8['end 1>2'].astype(bool).mean():.0%}, "
      f"P3 > P2 in {raw8['end 3>2'].astype(bool).mean():.0%}")
print("main reading:", pm[["end P1", "end P2", "end P3", "p end 1>2", "p end 3>2"]].round(4).to_dict())
print("ranges:", {k: (round(t8[k].min()), round(t8[k].max())) for k in ["end P1", "end P2", "end P3"]})

**Result: not reproduced.** At the most peripheral ROI Packet 1 gives 810, Packet 2 525 and
Packet 3 284 a.u. in the main reading; no reading puts the values in the expected ranges. Packet
3 is below Packet 2 (p = 0.0015 for a difference in the other direction), and Packet 1 is above
Packet 2 at p = 0.025, not p < 0.01; both tests hold in none of the 192 readings.

## fig8b-packet2-best-targeting

> Normalized to the most peripheral ROI, YP emission in the center is 183% for Packet 2 and
> drops to 83% for Packet 3 and 74% for Packet 1.

In [ ]:
nm = [f"norm P{k}" for k in G8]
exp8 = {"norm P1": 74, "norm P2": 183, "norm P3": 83}
t8["norm_in"] = np.all([(t8[k] - e).abs() <= 0.1 * e for k, e in exp8.items()], axis=0)
t8["norm_partial"] = (t8["norm P2"] > 100) & (t8["norm P1"] < 100) & (t8["norm P3"] < 100)
show_spread(t8, nm)
print(f"\nall three within 10% in {t8.norm_in.sum()} of {len(t8)} readings; Packet 2 above 100% and Packets 1 "
      f"and 3 below in {t8.norm_partial.sum()}")
print("ranges:", {k: (round(t8[k].min()), round(t8[k].max())) for k in nm})
print("main reading:", pm[nm].round(1).to_dict())
print("highest normalized centre, share of readings:", t8[nm].idxmax(axis=1).value_counts(normalize=True).round(3).to_dict())

A check of the labels. The session descriptions printed below define Protocol 2 as 8 phases
without delay and Protocol 3 as 8 phases with 5-us delays, which is Packet 2 and Packet 3 of
Methods 2.3 ("4 phases; 8 phases; and 8 phases with 5-us delays"). The outcomes use these labels.
The cell also runs the three Fig. 8 claims with the Protocol 2 and Protocol 3 labels exchanged,
as a variant with no support in the files.

In [ ]:
# What the files say each protocol is (session descriptions), against Methods 2.3: Packet 2 has
# 8 phases without delays, Packet 3 the same 8 phases with 5-us delays.
for k, v in G8.items():
    print(f"Protocol{k}:", sorted({r["description"] for r in v}))
G8s = {1: G8[1], 2: G8[3], 3: G8[2]}
rows8s = []
for rd in cc.iter_readings():
    rows8s.append({**rd, **fig8(rd, G8s)})
t8s = pd.DataFrame(rows8s)
t8s["norm_in"] = np.all([(t8s[k] - e).abs() <= 0.1 * e for k, e in exp8.items()], axis=0)
t8s["centres_in"] = ((t8s[cent] >= 193.5) & (t8s[cent] <= 258.5)).all(axis=1)
t8s["ends_in"] = (t8s["end P1"].between(225, 385) & t8s["end P3"].between(225, 385) & t8s["end P2"].between(108, 132))
print("labels exchanged: normalized centres within 10% in", int(t8s.norm_in.sum()), "readings; centres in range in",
      int(t8s.centres_in.sum()), "; ends in range in", int(t8s.ends_in.sum()), "of", len(t8s))
print("main reading, exchanged:", primary(t8s)[nm + cent + ["end P1", "end P2", "end P3"]].round(1).to_dict())
print("highest normalized centre, exchanged:", t8s[nm].idxmax(axis=1).value_counts(normalize=True).round(3).to_dict())
print("acquisition times:", {k: [(r["subject_id"], r["start"][:16]) for r in v] for k, v in G8.items()})

**Result: not reproduced.** The normalized centre is 88% for Packet 1, 73% for Packet 2 and 141%
for Packet 3 in the main reading. No reading puts the three within 10% of 74, 183 and 83%, and
Packet 3 has the highest normalized centre in every reading. The session descriptions confirm
the labels (Protocol 2 has 8 phases without delays, Protocol 3 the same with 5-us delays). With
the Protocol 2 and Protocol 3 labels exchanged, a variant the files do not support, Packet 2
would have the highest normalized centre in every reading and 19 readings would put all three
within 10%; the centre and near-electrode values in a.u. would still not match.

## fig9a-gap-reduces-periphery-more

> With the large array lifted 1, 2 and 3 mm above the monolayer, YP emission near the
> electrodes falls 2.4, 6.4 and 9 times against electrodes touching the cells, while in the
> center it falls only 1.02, 1.4 and 3.8 times.

In [ ]:
G9 = {g_: dishes("001028", gap_mm=g_) for g_ in (0.0, 1.0, 2.0, 3.0)}
print({k: [r["subject_id"] for r in v] for k, v in G9.items()})
EXP9 = {"end 1": 2.4, "end 2": 6.4, "end 3": 9, "centre 1": 1.02, "centre 2": 1.4, "centre 3": 3.8}


def fig9(rd):
    gv = {k: cc.group_values(v, rd) for k, v in G9.items()}
    res = {}
    for gap in (1.0, 2.0, 3.0):
        res[f"end {gap:.0f}"] = gv[0.0]["end"] / gv[gap]["end"]
        res[f"centre {gap:.0f}"] = gv[0.0]["centre"] / gv[gap]["centre"]
    for gap in (0.0, 1.0, 2.0, 3.0):
        res[f"norm {gap:.0f}"] = gv[gap]["norm_centre"]
    res["n_in"] = sum(abs(res[k] - e) <= 0.1 * e for k, e in EXP9.items())
    res["end>centre all gaps"] = all(res[f"end {g:.0f}"] > res[f"centre {g:.0f}"] for g in (1, 2, 3))
    return res


t9 = readings_table(fig9)
show_spread(t9, list(EXP9), by=("o1", "w", "o2", "o3", "o4", "o6", "o7"))
print(f"\n{len(t9)} readings; all six within 10% in {(t9.n_in == 6).sum()}; most within: {t9.n_in.max()}; "
      f"near-electrode ratio above centre ratio at every gap in {t9['end>centre all gaps'].mean():.0%}")
print("main reading:", primary(t9)[list(EXP9)].round(2).to_dict())
print("ranges:", {k: (round(t9[k].min(), 2), round(t9[k].max(), 2)) for k in EXP9})
print("best reading:")
print(t9.loc[t9.n_in.idxmax()].to_string())
gv9 = {k: cc.group_values(v, cc.PRIMARY) for k, v in G9.items()}
for k, g_ in gv9.items():
    print(f"{k:.0f} mm: end {g_['end']:.0f} (dishes {np.round(g_['dish_end']).tolist()}), centre {g_['centre']:.0f} "
          f"(dishes {np.round(g_['dish_centre']).tolist()}) a.u.")

**Result: partially reproduced.** In the main reading uptake near the electrodes falls 2.38,
4.68 and 8.20 times with 1, 2 and 3 mm gaps, and in the centre 1.06, 1.91 and 4.07 times (paper
2.4, 6.4, 9 and 1.02, 1.4, 3.8). No reading has all six ratios within 10% (at most 4 of 6, in the
main reading), and the near-electrode ratio exceeds the centre ratio at every gap in 88% of the
576 readings, including the main one.

## fig9b-gap-boosts-targeting

> The faster fading near the electrodes raises targeting efficiency (center over periphery) to
> as much as 600% with a 2-mm gap.

In [ ]:
norm9 = [f"norm {g:.0f}" for g in (0, 1, 2, 3)]
t9["in"] = t9["norm 2"].between(540, 660)
t9["largest"] = t9[norm9].idxmax(axis=1) == "norm 2"
show_spread(t9, norm9)
print(f"\n2-mm value in 540 to 660% in {t9['in'].sum()} of {len(t9)} readings; and the largest in "
      f"{(t9['in'] & t9['largest']).sum()}; in tolerance but another higher in {(t9['in'] & ~t9['largest']).sum()}")
print("largest distance, share of readings:", t9[norm9].idxmax(axis=1).value_counts(normalize=True).round(3).to_dict())
print("main reading:", primary(t9)[norm9].round(0).to_dict())
fig, ax = plt.subplots(1, 2, figsize=(11, 3.5))
for k, g_ in gv9.items():
    ax[0].plot(g_["d"], g_["profile"], label=f"{k:.0f} mm"); ax[1].plot(g_["d"], g_["norm_profile"], label=f"{k:.0f} mm")
ax[0].set_ylabel("YP, a.u."); ax[1].set_ylabel("% of most peripheral ROI"); ax[0].legend()
for a in ax: a.set_xlabel("distance from centre, mm")
plt.tight_layout(); plt.show()
print("plotted ranges:", {f"{k:.0f} mm": (round(float(np.nanmin(g_['norm_profile']))), round(float(np.nanmax(g_['norm_profile'])))) for k, g_ in gv9.items()}, "%")

**Result: reproduced.** The normalized centre at 2 mm is 541% in the main reading, the largest
of the four distances (198, 466, 541 and 382% at 0, 1, 2 and 3 mm). It is within 540 to 660% and
the largest in 107 of 576 readings, and within tolerance with another distance higher in 19.
The 2 mm value is the largest of the four in 84% of readings.

## fig9c-restricted-to-center

> With the electrodes placed above the monolayer, electroporation is restricted almost
> exclusively to the center of the array.

At 1, 2 and 3 mm: centre above the most peripheral ROI (paired and unpaired, O8), and the
centre the highest point of the mean profile.

In [ ]:
def fig9c(rd):
    res = {}
    for gap in (1.0, 2.0, 3.0):
        g_ = cc.group_values(G9[gap], rd)
        for pr in ("paired", "unpaired"):
            t = ttest(g_["dish_centre"], g_["dish_end"], paired=pr == "paired")
            res[f"p {gap:.0f} {pr}"] = t.pvalue
            res[f"c>end {gap:.0f} {pr}"] = (g_["centre"] > g_["end"]) and t.pvalue < 0.05
        res[f"max at centre {gap:.0f}"] = int(np.nanargmax(g_["profile"])) == 0
        res[f"max at mm {gap:.0f}"] = g_["d"][int(np.nanargmax(g_["profile"]))]
    return res


t9c = readings_table(fig9c, o5="a", o7="raw")
tests9 = [c for c in t9c.columns if c.startswith("c>end")]
mx = [c for c in t9c.columns if c.startswith("max at centre")]
t9c["tests unpaired"] = t9c[[c for c in tests9 if c.endswith("unpaired")]].all(axis=1)
t9c["tests paired"] = t9c[[c for c in tests9 if c.endswith(" paired")]].all(axis=1)
t9c["max"] = t9c[mx].all(axis=1)
for c in tests9 + mx + ["tests unpaired", "tests paired", "max"]:
    print(f"{c}: holds in {t9c[c].mean():.0%} of {len(t9c)} readings")
pm = primary(t9c)
print("main reading:", pm[[c for c in t9c.columns if c.startswith(("p ", "max at mm"))]].round(5).to_dict())
show_spread(t9c, [c for c in t9c.columns if c.startswith("max at mm")], by=("o1", "w", "o2", "o3", "o6"))
OUT9C_T = decide(t9c, "tests paired", "centre above the most peripheral ROI at 1, 2 and 3 mm, O8 paired")
OUT9C_M = decide(t9c, "max", "centre the highest point at 1, 2 and 3 mm")
print(supported(t9c)[["o1", "o2", "o4"] + [c for c in t9c.columns if c.startswith("p ") and c.endswith(" paired")] + ["tests paired", "max"]].round(4).to_string())

**Result: not reproduced.** By the paired test the centre is above the most peripheral ROI at
1 mm (p = 0.021) but not at 2 mm (p = 0.051) or 3 mm (p = 0.34) in the main reading, and at all
three gaps in 8% of readings and in none of the 12 supported ones, with O2 varied (unpaired: 25%
of readings). At 2 mm alone the paired test holds in 92% of readings. The
centre is the highest point of the mean profile at all three gaps in none of the 96 readings;
in the main reading the maximum at 2 mm is at 0.44 mm from the centre. One of the three 3 mm
dishes has a centre value of 3782 a.u. against 592 and 660 in the other two.

## Summary

A failure to reproduce here may be a failure of this attempt, not of the paper: see the
reasoning and the differences from the paper's method.

| Claim | Outcome |
|---|---|
| fig3-field-simulation | blocked, data_missing |
| fig4-sham-no-uptake | blocked, data_missing |
| fig4-unipolar-strongest | blocked, data_missing |
| fig4-bipolar-weaker | blocked, data_missing |
| fig4-twenty-fold-cancellation | blocked, data_missing |
| fig5a-trains-stronger | not reproduced |
| fig5a-packet-center-enhanced | partially reproduced |
| fig5b-rotation-center-peak | blocked, methods_underspecified (the supported readings disagree) |
| fig5c-normalized-contrast | blocked, depends on fig5b-rotation-center-peak (its own test holds) |
| fig6a-more-packets-more-uptake | partially reproduced |
| fig6b-peak-relative-to-periphery | partially reproduced |
| fig6-trough-half-of-center | partially reproduced |
| fig6-peak-narrower | blocked, methods_underspecified (the supported readings disagree) |
| fig7a-amplitude-scales-uptake | partially reproduced |
| fig7b-targeting-independent-of-amplitude | blocked, methods_underspecified (the supported readings disagree) |
| fig7cd-holds-for-large-array | blocked, methods_underspecified (the supported readings disagree) |
| discussion-independent-of-array-size | blocked, depends on fig7b-targeting-independent-of-amplitude |
| fig8a-equal-center-uptake | not reproduced |
| fig8a-more-uptake-near-electrodes | not reproduced |
| fig8b-packet2-best-targeting | not reproduced |
| fig8-cancellation-predicts-targeting | blocked, data_missing |
| fig9a-gap-reduces-periphery-more | partially reproduced |
| fig9b-gap-boosts-targeting | reproduced |
| fig9c-restricted-to-center | not reproduced |

How the analysis differs from the paper's. The paper measured ROIs in MetaMorph on images whose
electrode positions and ROIs are not deposited; here the electrode square is found in each image
and the ROIs are rebuilt from Methods 2.4 under the claims file's readings, with a pixel size of
0.65 um assumed (it gives a large-array diagonal of 8.25 mm between needle edges, against the
paper's 8.4 mm). For electrodes 2 and 3 mm above the cells the square comes from needle marks in
the second channel. Where the readings of a test disagree, the outcome is the one the supported
readings give, with comparisons inside a dish paired; where those disagree too, the claim is
blocked: fig5b-rotation-center-peak and fig7b-targeting-independent-of-amplitude, both split by
the ROI count, and fig7b also by the end ROI and the diagonals; fig6-peak-narrower and
fig7cd-holds-for-large-array, split by the diagonals. Both diagonals are fixed only for Fig. 5,
whose legend gives them; for Figs. 6 to 9 the claims file says the paper does not say, so the
diagonals are varied. Values in a.u. are the files' pixel values.
The paper's Fig. 8 centres, 215 to 235 a.u., are below the 378 to 692 a.u. here in raw intensity
and overlap the 154 to 418 a.u. with background subtracted; whether its scale differs from the
files' is asked of a reader.

What was found about the datasets. No dataset holds the single-cell time lapse of Fig. 4 or the
field maps of Fig. 3. The files give no pixel size, ROI, electrode position or e1 position, and
000686 and 000875 give the species as Bos taurus for CHO cells. Six dishes are deposited twice,
in two dandisets (000875 and 001025; 001025 and 001028), consistent with their use in two
figures. 001025 has 2 large-array dishes at 4.8 kV and 4 small-array dishes at 2.4 kV, where the
paper gives n = 3. In 001027 the dishes labelled Protocol 3, which the session descriptions
define as Packet 3 (8 phases with 5-us delays), give the highest normalized centre, the targeting
the paper reports for Packet 2.

In [ ]:
print(f"total run time {(time.time() - T0) / 60:.1f} min; peak memory "
      f"{resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / 1e6:.1f} GB")